# SupportPulse AI: Exploratory Data Science & Predictive Analytics
### From Support Tickets to Actionable Intelligence

This notebook documents the full analytical & ML pipeline of **SupportPulse AI**:
- **Data Ingestion & Quality Validation** (missing value treatment, duplicate detection, IQR outlier bounds)
- **Support Friction Index** (composite weighted metric: response delay, resolution delay, priority severity, satisfaction penalty)
- **Satisfaction Drop-Point Analysis** (empirical detection of CSAT cliff at response threshold)
- **Team Benchmarking with Complexity Normalization** (Expected resolution baseline based on Category + Priority)
- **NLP Recurring Theme Discovery** (TF-IDF + Clustering for hidden failure modes)
- **Multinomial Naive Bayes Machine Learning** (Predicting Satisfaction Risk: High, Medium, Low)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv('../data/raw/supportpulse_ai_support_tickets_25000.csv')
print('Loaded raw shape:', df.shape)
df.head()

## 1. Data Cleaning & Validation

In [ ]:
# Remove duplicates
df = df.drop_duplicates(subset=['Ticket_ID'])
# Remove negative/invalid times
valid = (df['Response_Time'] >= 0) & (df['Resolution_Time'] >= 0) & df['Satisfaction_Score'].between(1, 5)
df = df[valid].copy()
# Impute missing
df['Description'] = df['Description'].fillna('Unknown')
df['Team'] = df['Team'].fillna('Unknown')
print('Cleaned rows:', len(df))

## 2. Support Friction Score
Formula:
$$\text{Friction} = 0.30 \times \text{RespDelay} + 0.30 \times \text{ResDelay} + 0.20 \times \text{Priority} + 0.20 \times \text{SatPenalty}$$
Tiers: Low (0.00-0.30), Moderate (0.31-0.60), Critical (0.61-1.00)

In [ ]:
prio_map = {'Low': 0.1, 'Medium': 0.4, 'High': 0.7, 'Urgent': 1.0}
prio_norm = df['Priority'].map(prio_map).fillna(0.4)
resp_norm = np.clip(df['Response_Time'] / df['Response_Time'].quantile(0.99), 0, 1)
res_norm = np.clip(df['Resolution_Time'] / df['Resolution_Time'].quantile(0.99), 0, 1)
sat_pen = (5.0 - df['Satisfaction_Score']) / 4.0

df['Friction_Score'] = (0.30 * resp_norm + 0.30 * res_norm + 0.20 * prio_norm + 0.20 * sat_pen).round(3)
print(df['Friction_Score'].describe())

## 3. Team Complexity Baseline & Anomaly Detection

In [ ]:
baseline = df.groupby(['Category', 'Priority'])['Resolution_Time'].median().rename('Expected_Res')
df = df.merge(baseline, on=['Category', 'Priority'], how='left')
df['Resolution_Deviation'] = df['Resolution_Time'] - df['Expected_Res']
team_summary = df.groupby('Team')[['Resolution_Time', 'Expected_Res', 'Resolution_Deviation']].mean()
team_summary['status'] = team_summary['Resolution_Deviation'].apply(lambda d: 'Anomalous' if d > 5.0 else 'Normal')
team_summary

## 4. Naive Bayes Satisfaction Risk Classification

In [ ]:
def map_risk(score):
    if score <= 2: return 'High Risk'
    elif score == 3: return 'Medium Risk'
    else: return 'Low Risk'

df['Satisfaction_Risk'] = df['Satisfaction_Score'].apply(map_risk)
X = df[['Description', 'Category', 'Priority', 'Team', 'Response_Time']]
y = df['Satisfaction_Risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
prep = ColumnTransformer([
    ('text', TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=3000), 'Description'),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['Category', 'Priority', 'Team']),
    ('num', MinMaxScaler(), ['Response_Time'])
])
pipe = Pipeline([('prep', prep), ('clf', MultinomialNB(alpha=0.2, fit_prior=False))])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))